# 08 - Colab Public URL Demo Test / URL 图片公开 Demo 测试

This standalone notebook retests the public Gradio demo flow with direct online image URLs. It is meant for interview/demo rehearsal after the final portfolio validation notebook.

这个独立 notebook 用于重新测试公开 Gradio demo：支持上传图片，也支持粘贴网上直接图片 URL。主页面只展示 compact summary、检测表格和短报告；完整 Prediction JSON 保留在默认折叠的 debug 区，并在打印/PDF 时隐藏。

## Notes / 注意

- Use a direct image URL, such as a `.jpg`, `.png`, or `.webp` address copied from “open image in new tab”.
- A normal web page URL may fail because it returns HTML instead of image bytes.
- The URL smoke cell is optional: leave `TEST_IMAGE_URL = ""` to skip it.
- The final Gradio launch cell is intentionally long-running and should be run manually when you want a public URL.

In [ ]:
# CN: 挂载 Drive 并设置路径。
# EN: Mount Drive and configure shared paths.
from google.colab import drive
from pathlib import Path
import os

drive.mount('/content/drive')

DRIVE_ROOT = Path('/content/drive/MyDrive/CarDD_YOLO11')
REPO_URL = 'https://github.com/lsdlBlueDragon/ai-powered-vehicle-damage-assessment-pipeline.git'
REPO_CLONE_DIR = Path('/content/ai-powered-vehicle-damage-assessment-pipeline')

YOLO_WEIGHTS = DRIVE_ROOT / 'runs/train/yolo11n_seg/weights/best.pt'
QWEN_ADAPTER_DIR = DRIVE_ROOT / 'llm_adapters/qwen2_5_7b_cardd_report_lora'
URL_INPUT_DIR = DRIVE_ROOT / 'demo_url_inputs'
URL_DEMO_OUTPUT_DIR = DRIVE_ROOT / 'runs/predict/url_demo_smoke'
URL_DEMO_SMOKE_JSON = DRIVE_ROOT / 'reports/url_demo_smoke_output.json'
GRADIO_OUTPUT_DIR = DRIVE_ROOT / 'runs/gradio_url_demo'

for key, value in {
    'DRIVE_ROOT': DRIVE_ROOT,
    'YOLO_WEIGHTS': YOLO_WEIGHTS,
    'QWEN_ADAPTER_DIR': QWEN_ADAPTER_DIR,
    'URL_INPUT_DIR': URL_INPUT_DIR,
    'URL_DEMO_OUTPUT_DIR': URL_DEMO_OUTPUT_DIR,
    'URL_DEMO_SMOKE_JSON': URL_DEMO_SMOKE_JSON,
    'GRADIO_OUTPUT_DIR': GRADIO_OUTPUT_DIR,
}.items():
    print(f'{key}: {value}')

In [ ]:
# CN: 同步最新轻量仓库代码到 Drive，不覆盖数据、权重、adapter 或 runs。
# EN: Sync latest lightweight repo code into Drive without touching heavy artifacts.
%cd /content
!apt-get update -qq && apt-get install -y -qq rsync
!rm -rf "$REPO_CLONE_DIR"
!git clone --depth 1 "$REPO_URL" "$REPO_CLONE_DIR"

for folder in ['src', 'scripts', 'tests', 'docs', 'configs', 'notebooks']:
    src = REPO_CLONE_DIR / folder
    dst = DRIVE_ROOT / folder
    dst.mkdir(parents=True, exist_ok=True)
    !rsync -a --delete "$src/" "$dst/"

for file_name in ['README.md', 'requirements-colab.txt', 'pyproject.toml']:
    src = REPO_CLONE_DIR / file_name
    if src.exists():
        !cp "$src" "$DRIVE_ROOT/$file_name"

expected_markers = [
    (DRIVE_ROOT / 'src/vehicle_damage_pipeline/service/colab_public_demo.py', 'resolve_image_input'),
    (DRIVE_ROOT / 'src/vehicle_damage_pipeline/service/colab_public_demo.py', 'build_demo_css'),
    (DRIVE_ROOT / 'tests/test_colab_public_demo.py', 'test_demo_css_hides_debug_json_when_printing'),
    (DRIVE_ROOT / 'notebooks/08_colab_public_url_demo_test.ipynb', 'URL 图片公开 Demo 测试'),
]
for path, marker in expected_markers.items():
    assert path.exists(), f'Missing synced file: {path}'
    text = path.read_text(encoding='utf-8', errors='ignore')
    assert marker in text, f'Missing marker {marker!r} in {path}. Sync may be stale.'
    assert '?' * 3 not in text, f'Encoding corruption detected in {path}'

print('Lightweight sync complete. / 轻量代码同步完成。')

In [ ]:
# CN: 安装 Colab 依赖和 editable package。
# EN: Install Colab dependencies and editable package.
%cd /content/drive/MyDrive/CarDD_YOLO11
%pip install -q -U pip
%pip install -q gdown
%pip install -q -e ".[vision,service,dev]"

pyproject_text = (DRIVE_ROOT / 'pyproject.toml').read_text(encoding='utf-8')
for marker in ['vision = [', 'service = [', 'dev = [']:
    assert marker in pyproject_text, f'Missing dependency extra marker: {marker}'
print('Dependency extras verified. / 依赖 extras 已验证。')

In [ ]:
# CN: 检查 demo 所需产物。
# EN: Check required demo artifacts.
required = {
    'YOLO weights': YOLO_WEIGHTS,
    'colab_public_demo.py': DRIVE_ROOT / 'src/vehicle_damage_pipeline/service/colab_public_demo.py',
    'launcher script': DRIVE_ROOT / 'scripts/colab_gradio_public_demo.py',
}
missing = []
for label, item in required.items():
    if item.exists():
        print(f'OK {label}: {item}')
    else:
        print(f'MISSING {label}: {item}')
        missing.append(str(item))
assert not missing, missing

In [ ]:
# CN: 验证 URL helper 和 debug JSON 折叠/打印隐藏逻辑。
# EN: Verify URL helpers and print-hidden debug JSON behavior.
%cd /content/drive/MyDrive/CarDD_YOLO11
!python -m pytest tests/test_colab_public_demo.py tests/test_demo_display.py -q -p no:cacheprovider

from pathlib import Path
import importlib
import os
import select
import subprocess
import sys
import time

DRIVE_ROOT = Path(globals().get('DRIVE_ROOT', '/content/drive/MyDrive/CarDD_YOLO11'))
src_root = DRIVE_ROOT / 'src'
if str(src_root) not in sys.path:
    sys.path.insert(0, str(src_root))
importlib.invalidate_caches()

try:
    import vehicle_damage_pipeline.service.colab_public_demo as demo_module
except ModuleNotFoundError:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', '-e', str(DRIVE_ROOT)], check=True)
    importlib.invalidate_caches()
    import vehicle_damage_pipeline.service.colab_public_demo as demo_module

print('Using demo module:', demo_module.__file__)
build_demo_css = demo_module.build_demo_css
download_image_url = demo_module.download_image_url
resolve_image_input = demo_module.resolve_image_input

css = build_demo_css()
assert 'debug-json-panel' in css
assert '@media print' in css
assert 'display: none' in css
print('URL helper and folded debug panel checks passed. / URL helper 和折叠 debug 区检查通过。')

In [ ]:
# CN: 可选：填入一个网上直接图片 URL，下载并跑一次单图预测 smoke test。
# EN: Optional: paste a direct online image URL and run one prediction smoke test.
import json
from pathlib import Path

from vehicle_damage_pipeline.service.colab_public_demo import download_image_url
from vehicle_damage_pipeline.service.display import (
    build_debug_prediction_json,
    build_detection_table,
    build_public_prediction_summary,
)
from vehicle_damage_pipeline.service.predictor import DamagePredictor, generate_assessment_report

# Paste a direct image URL here. Leave empty to skip this optional smoke test.
TEST_IMAGE_URL = ""

if TEST_IMAGE_URL.strip():
    downloaded = Path(download_image_url(TEST_IMAGE_URL, URL_INPUT_DIR))
    print('Downloaded URL image:', downloaded)

    predictor = DamagePredictor(weights=YOLO_WEIGHTS, output_dir=URL_DEMO_OUTPUT_DIR)
    prediction = predictor.predict_file(downloaded)
    public_summary = build_public_prediction_summary(prediction)
    detection_table = build_detection_table(prediction)
    report = generate_assessment_report(prediction, backend='template')
    debug_json = build_debug_prediction_json(prediction)

    public_text = json.dumps(public_summary, ensure_ascii=False)
    assert 'mask_polygon' not in public_text
    assert 'mask_polygon' in debug_json or not prediction.get('detections')

    URL_DEMO_SMOKE_JSON.parent.mkdir(parents=True, exist_ok=True)
    URL_DEMO_SMOKE_JSON.write_text(
        json.dumps(
            {
                'test_image_url': TEST_IMAGE_URL,
                'downloaded_image': str(downloaded),
                'public_summary': public_summary,
                'detection_table': detection_table,
                'assessment_report': report,
                'debug_json_has_mask_polygon': 'mask_polygon' in debug_json,
            },
            ensure_ascii=False,
            indent=2,
        ),
        encoding='utf-8',
    )
    print(json.dumps(public_summary, ensure_ascii=False, indent=2)[:2000])
    print(detection_table)
    print(report)
    print('URL smoke output:', URL_DEMO_SMOKE_JSON)
else:
    print('TEST_IMAGE_URL is empty; skipped URL prediction smoke test. Paste a direct image URL and rerun this cell when needed.')

## Launch Public Demo / 启动公开 Demo

Run the next cell manually when you want a public `gradio.live` URL. The demo supports both uploaded images and direct online image URLs. It uses `share=True` through `build_colab_launch_kwargs()`.

默认使用 `--no-qwen` 让网页 demo 更轻、更稳定；如果想测试 Qwen adapter 报告，把 `USE_QWEN_REPORT = True` 后重新运行。

In [ ]:
# CN: 手动启动公开 Gradio demo。此 cell 会持续运行，停止 cell 即可关闭链接。
# EN: Manually launch the public Gradio demo. This cell is long-running.
import subprocess
import sys

USE_QWEN_REPORT = False
SERVER_PORT = 7860
STARTUP_TIMEOUT_SECONDS = 240

cmd = [
    sys.executable,
    '-u',
    'scripts/colab_gradio_public_demo.py',
    '--weights',
    str(YOLO_WEIGHTS),
    '--output-dir',
    str(GRADIO_OUTPUT_DIR),
    '--sample-dir',
    str(DRIVE_ROOT / 'demo_upload_samples'),
    '--adapter-dir',
    str(QWEN_ADAPTER_DIR),
    '--server-port',
    str(SERVER_PORT),
]
if not USE_QWEN_REPORT:
    cmd.append('--no-qwen')

env = os.environ.copy()
env['PYTHONUNBUFFERED'] = '1'

print('Launching URL-capable public demo. The script uses share=True.')
print(' '.join(cmd))
print('Waiting for Gradio output. This cell stays running while the public demo is alive.')

process = subprocess.Popen(
    cmd,
    cwd=DRIVE_ROOT,
    env=env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

started = time.time()
saw_url = False
try:
    while True:
        return_code = process.poll()
        ready, _, _ = select.select([process.stdout], [], [], 5)
        if ready:
            line = process.stdout.readline()
            if line:
                print(line, end='')
                if 'Running on public URL:' in line or 'Running on local URL:' in line:
                    saw_url = True
        elif return_code is None:
            elapsed = int(time.time() - started)
            if not saw_url and elapsed and elapsed % 30 == 0:
                print(f'[startup] still waiting for Gradio URL... {elapsed}s elapsed')

        if return_code is not None:
            remaining = process.stdout.read() if process.stdout else ''
            if remaining:
                print(remaining, end='')
            if return_code != 0:
                raise subprocess.CalledProcessError(return_code, cmd)
            break

        if not saw_url and time.time() - started > STARTUP_TIMEOUT_SECONDS:
            process.terminate()
            print('\nTimed out before Gradio printed a local/public URL.')
            print('Try rerunning with SERVER_PORT = 7861, or set USE_QWEN_REPORT = False first to isolate Gradio share startup.')
            raise TimeoutError('Gradio did not print a URL before startup timeout.')
except KeyboardInterrupt:
    process.terminate()
    print('\nStopped Gradio demo process.')
    raise

## Result / 结果

When the helper checks pass and the public demo launches, use the generated public URL to test either an uploaded image or a direct online image URL. For damaged images, the page keeps the full Prediction JSON in a collapsed debug panel and hides it during browser print/PDF output.